In [1]:
import pandas as pd
import numpy as np

In [15]:
# =============================================================================
# Calculate Portion Weights for ASA24_GPTFoodCodes_portion.csv
# Uses FoodWeights.csv to look up standard portion weights and multiply by amount
# =============================================================================

import re

# Load the files
df_portion_results = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_portion.csv')
df_weights = pd.read_csv('../data/FoodWeights.csv')

# Extract base unit from FoodWeights portions (e.g., "1 cup" -> "cup", "1 tablespoon" -> "tablespoon")
def extract_base_unit(portion):
    """Extract the base unit from a portion string like '1 cup' or '1 cup, cooked'"""
    if pd.isna(portion):
        return None
    portion = str(portion).lower().strip()
    match = re.match(r'^1\s+([a-z]+)', portion)
    if match:
        return match.group(1)
    return None

# Build the weight lookup table
# For each FoodCode + unit combination, keep the simplest (shortest) portion description
df_weights['BaseUnit'] = df_weights['Portion'].apply(extract_base_unit)
df_weights['Simplicity'] = df_weights['Portion'].apply(lambda x: len(str(x)) if pd.notna(x) else 999)

df_weights_simple = df_weights.dropna(subset=['BaseUnit']).sort_values('Simplicity')
df_weights_simple = df_weights_simple.drop_duplicates(subset=['FoodCode', 'BaseUnit'], keep='first')

weight_lookup = df_weights_simple[['FoodCode', 'BaseUnit', 'Portion weight (g)']].copy()

print(f"Weight lookup table: {len(weight_lookup)} entries")
print(f"Unique FoodCodes in lookup: {weight_lookup['FoodCode'].nunique()}")
print(f"Unique BaseUnits in lookup: {weight_lookup['BaseUnit'].nunique()}")

Weight lookup table: 14937 entries
Unique FoodCodes in lookup: 5572
Unique BaseUnits in lookup: 391


In [ ]:
# Normalize ASA24 LabelUnit to match FoodWeights BaseUnit
df_portion_results['BaseUnit'] = df_portion_results['LabelUnit'].str.lower().str.strip()

# Merge to get standard portion weights (ground truth unit)
df_portion_merged = df_portion_results.merge(
    weight_lookup,
    left_on=['FoodCode', 'BaseUnit'],
    right_on=['FoodCode', 'BaseUnit'],
    how='left'
)

# Calculate ground truth portion weight (uses LabelUnit + LabelAmount)
df_portion_merged['CalculatedWeight'] = (
    df_portion_merged['Portion weight (g)'] * df_portion_merged['LabelAmount']
)

# ---------------------------------------------------------------------------
# GPT weight calculation: use GPTPortionDescription unit + GPTPortionAmount
# ---------------------------------------------------------------------------

def extract_gpt_unit(description):
    """Extract normalized base unit from GPTPortionDescription.

    Handles any numeric prefix (not just '1') and strips plurals.
    Examples: '2 cups' -> 'cup', '1 tablespoon' -> 'tablespoon'
    Returns None for unrecognizable / failure strings.
    """
    if pd.isna(description):
        return None
    text = str(description).lower().strip()
    match = re.match(r'^\d+(?:\.\d+)?\s+([a-z]+)', text)
    if not match:
        return None
    unit = match.group(1)
    # Strip trailing 's' to normalise plurals (cups->cup, pieces->piece)
    # Avoid over-stripping short words like 'oz'
    if len(unit) > 3 and unit.endswith('s'):
        unit = unit[:-1]
    return unit

# Extract GPT's chosen unit from GPTPortionDescription
df_portion_merged['GPTBaseUnit'] = (
    df_portion_merged['GPTPortionDescription'].apply(extract_gpt_unit)
)

# Separate weight lookup using GPT's unit (not LabelUnit)
gpt_weight_lookup = weight_lookup.rename(columns={'Portion weight (g)': 'Portion weight (g)_GPT'})
df_portion_merged = df_portion_merged.merge(
    gpt_weight_lookup,
    left_on=['FoodCode', 'GPTBaseUnit'],
    right_on=['FoodCode', 'BaseUnit'],
    how='left',
    suffixes=('', '_gpt_lookup')
).drop(columns=['BaseUnit_gpt_lookup'], errors='ignore')

# GPT weight = GPT weight-per-unit x GPT predicted amount
df_portion_merged['CalculatedWeightGPT'] = (
    df_portion_merged['Portion weight (g)_GPT'] *
    pd.to_numeric(df_portion_merged['GPTPortionAmount'], errors='coerce')
)

# ---------------------------------------------------------------------------
# Match statistics
# ---------------------------------------------------------------------------
total_rows = len(df_portion_merged)
matched_rows = df_portion_merged['Portion weight (g)'].notna().sum()
unmatched_rows = df_portion_merged['Portion weight (g)'].isna().sum()
gpt_matched = df_portion_merged['Portion weight (g)_GPT'].notna().sum()

print(f"=== MATCH RESULTS ===")
print(f"Total rows:         {total_rows}")
print(f"GT unit matched:    {matched_rows} ({matched_rows/total_rows*100:.1f}%)")
print(f"GT unit unmatched:  {unmatched_rows} ({unmatched_rows/total_rows*100:.1f}%)")
print(f"GPT unit matched:   {gpt_matched} ({gpt_matched/total_rows*100:.1f}%)")
print(f"GPT unit unmatched: {total_rows - gpt_matched} ({(total_rows-gpt_matched)/total_rows*100:.1f}%)")

In [20]:
df_matched_only = df_portion_merged[df_portion_merged['Portion weight (g)'].notna()]
df_matched_only.head()

,Unnamed: 0,FileName,PortionType,FoodCode,Main Food description,Portion,Multiplier,Rank,PortionCode,PortionSubCode,...,LabelUnit,PortionShot,GPTPortionDescription,GPTPortionReason,GPTPortionAmount,GPTPortionAmountReason,BaseUnit,Portion weight (g),CalculatedWeight,CalculatedWeightGPT
0,0,cp_42fo.png,largest,94000100,"Water, tap",42 FO,42.0,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",16 FO,NaN,0,NaN,fo,30.0,1260.0,0.0
1,1,tcup_6fo.png,median,94000100,"Water, tap",6 FO,6.0,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",6 FO,NaN,0,NaN,fo,30.0,180.0,0.0
2,2,mug_2fo.png,smallest,94000100,"Water, tap",2 FO,0.1,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",I can't help to analyze this image.,"The mug's contents are not visible, so the por...",1,NaN,fo,30.0,60.0,30.0
3,3,cp_42fo.png,largest,94100100,"Water, bottled, plain",42 FO,42.0,1,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",8 FO,NaN,1,NaN,fo,30.0,1260.0,30.0
4,4,glsstall_12fo.png,median,94100100,"Water, bottled, plain",12 FO,6.0,1,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",I can't help to analyze this image.,Reason: the image shows an empty clear tall dr...,0,NaN,fo,30.0,360.0,0.0


In [21]:
df_matched_only.shape

(1055, 27)

In [13]:
df_matched_only.to_csv('../data/ASA24_GPTFoodCodes_portion_with_weights.csv', index=False)

In [3]:
# =============================================================================
# Calculate MAE (Mean Absolute Error) for DietAI24 on ASA24 dataset
# Per dish analysis for Energy, Fat, Carbohydrate, Protein, and Food Weight
# =============================================================================

import numpy as np

# Load the nutrition results file
df_nutrition = pd.read_csv('../data/ASA24_GPTFoodCodes_nutrition.csv')

print(f"Total rows in nutrition file: {len(df_nutrition)}")

# Exclude beverage reference images (fluid ounce portions = container reference images,
# e.g. foam cups, mugs, milk bottles used as portion size references, not food photos)
if 'Portion' in df_nutrition.columns:
    beverage_mask = df_nutrition['Portion'].str.contains('FO', na=False)
    print(f"Excluded {beverage_mask.sum()} beverage reference image rows (FO portions)")
    df_nutrition = df_nutrition[~beverage_mask].copy()
    print(f"Rows after exclusion: {len(df_nutrition)}")

# Filter to common images only (present in both old and new pipelines)
old_filenames = set(pd.read_csv('../data/archive/ASA24_GPTFoodCodes_nutrition_old.csv')['FileName'])
n_before = len(df_nutrition)
df_nutrition = df_nutrition[df_nutrition['FileName'].isin(old_filenames)].copy()
print(f"Filtered to {len(df_nutrition)} rows from common images (removed {n_before - len(df_nutrition)} new-image rows)")
print(f"Columns: {df_nutrition.columns.tolist()[:10]}...")  # Show first 10 columns

# Define the nutrients we're interested in
nutrients = ['Energy (kcal)', 'Protein (g)', 'Carbohydrate (g)', 'Total Fat (g)']

# Filter to only valid rows (where both GT and GPT predictions exist)
valid_rows = df_nutrition.copy()

# For each nutrient, filter to rows where both GT and prediction are not NaN
mae_results = {}

for nutrient in nutrients:
    gt_col = nutrient
    pred_col = f'{nutrient}_GPT'
    
    # Check if columns exist
    if gt_col not in df_nutrition.columns or pred_col not in df_nutrition.columns:
        print(f"Warning: {gt_col} or {pred_col} not found in dataframe")
        continue
    
    # Filter to valid rows for this nutrient (exclude zero predictions = GPT estimation failures)
    valid_mask = (
        df_nutrition[gt_col].notna() & 
        df_nutrition[pred_col].notna() &
        (df_nutrition[pred_col] != 0)
    )
    
    valid_data = df_nutrition[valid_mask]
    
    if len(valid_data) == 0:
        print(f"No valid data for {nutrient}")
        continue
    
    # Calculate MAE
    mae = np.abs(valid_data[gt_col] - valid_data[pred_col]).mean()
    
    mae_results[nutrient] = {
        'MAE': mae,
        'N': len(valid_data)
    }
    
# Calculate MAE for food weight
if 'CalculatedWeight' in df_nutrition.columns and 'CalculatedWeightGPT' in df_nutrition.columns:
    n_zero_weight = (df_nutrition['CalculatedWeightGPT'] == 0).sum()
    print(f"Excluded {n_zero_weight} rows with CalculatedWeightGPT=0 (GPT estimation failures)")
    weight_valid_mask = (
        df_nutrition['CalculatedWeight'].notna() & 
        df_nutrition['CalculatedWeightGPT'].notna() &
        (df_nutrition['CalculatedWeightGPT'] != 0)
    )
    weight_valid = df_nutrition[weight_valid_mask]
    
    if len(weight_valid) > 0:
        weight_mae = np.abs(weight_valid['CalculatedWeight'] - weight_valid['CalculatedWeightGPT']).mean()
        mae_results['Food weight (g)'] = {
            'MAE': weight_mae,
            'N': len(weight_valid)
        }

print("\n" + "="*70)
print("MAE Results for DietAI24 on ASA24 Dataset (Per Dish)")
print("="*70)
print(f"{'Target':<25} {'MAE (Per dish)':<20} {'N samples':<15}")
print("-"*70)

for target, results in mae_results.items():
    print(f"{target:<25} {results['MAE']:<20.2f} {results['N']:<15}")

print("="*70)

Total rows in nutrition file: 690
Excluded 0 beverage reference image rows (FO portions)
Rows after exclusion: 690
Filtered to 690 rows from common images (removed 0 new-image rows)
Columns: ['Unnamed: 0', 'FileName', 'PortionType', 'FoodCode', 'Main Food description', 'Portion', 'Multiplier', 'Rank', 'PortionCode', 'PortionSubCode']...
Excluded 10 rows with CalculatedWeightGPT=0 (GPT estimation failures)

MAE Results for DietAI24 on ASA24 Dataset (Per Dish)
Target                    MAE (Per dish)       N samples      
----------------------------------------------------------------------
Energy (kcal)             79.57                639            
Protein (g)               3.80                 617            
Carbohydrate (g)          8.72                 613            
Total Fat (g)             3.80                 620            
Food weight (g)           51.15                640            


In [2]:
df = pd.read_csv('../nutrition5k_final.csv')
df.head()

,dish_id,total_calories,total_mass,total_fat,total_carbs,total_protein,ingredients,sorted_ingredients,GPTFoodDescription,GPTFoodCode,GPTAmount,GPTAmountWeight,Energy (kcal),Protein (g),Carbohydrate (g),Total Fat (g),FoodCodeError,WeightError,IngredientNum,GPTWeight
0,dish_1567023335,138.336395,117,12.019873,5.832454,3.287284,"greek salad, garlic, salt, pecans, raspberries...","almonds, arugula, blueberries, carrot, chard, ...",curly kale leaves\narugula leaves\nSwiss chard...,curly kale leaves: 72119190; arugula leaves: 7...,curly kale leaves: 1 cup\narugula leaves: 0.5 ...,"{'curly kale leaves': 25.0, 'arugula leaves': ...",97.624,5.33878,5.29628,6.720720,0,0,6,114.80
1,dish_1561404956,97.600792,82,2.638521,4.593264,13.654346,"green onions, white rice, vinegar, ginger, par...","broccoli, caesar dressing, chicken, croutons, ...",roasted chicken thigh (skin-on)\nsautéed kale\...,roasted chicken thigh (skin-on): 24152230; sau...,roasted chicken thigh (skin-on): unknown\nsaut...,"{'roasted chicken thigh (skin-on)': nan, 'saut...",114.150,5.59750,6.84350,7.191500,0,1,3,140.00
2,dish_1563220109,184.735962,174,7.398599,22.408951,8.018723,"pepperoni pizza, salt, olive oil, squash, broc...","broccoli, olive oil, pepperoni pizza, salt, sq...",pepperoni (sliced)\nmozzarella cheese (melted)...,pepperoni (sliced): 25221250; mozzarella chees...,pepperoni (sliced): 1 slice\nmozzarella cheese...,"{'pepperoni (sliced)': 2.0, 'mozzarella cheese...",252.365,16.68025,15.51350,14.329600,0,1,6,213.00
3,dish_1561662501,158.395508,165,7.063837,21.929186,6.882387,"olive oil, pepper, corn on the cob, vinegar, c...","cherry tomatoes, corn on the cob, cucumbers, l...",cubed tofu\nbaby spinach leaves\nhalved cherry...,cubed tofu: 41420010; baby spinach leaves: 721...,cubed tofu: unknown\nbaby spinach leaves: unkn...,"{'cubed tofu': nan, 'baby spinach leaves': nan...",47.575,1.88075,9.94300,0.741875,0,2,4,151.25
4,dish_1565036821,325.539917,134,14.517972,25.940332,21.035995,"soy sauce, salt, vinegar, olive oil, beef, whi...","beef, broccoli, cheese pizza, chili, garlic, g...",melted mozzarella cheese\ntomato pizza sauce\n...,melted mozzarella cheese: 14107010; tomato piz...,"melted mozzarella cheese: 0.5 cup, melted\ntom...","{'melted mozzarella cheese': 122.0, 'tomato pi...",885.840,84.28930,23.96460,49.224100,0,2,5,405.00


In [6]:
# =============================================================================
# Calculate MAE (Mean Absolute Error) for DietAI24 on Nutrition5k dataset
# Per dish analysis for Energy, Fat, Carbohydrate, Protein, and Food Weight
# =============================================================================

print(f"Total rows in Nutrition5k results: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Define the mapping between Nutrition5k ground truth and FNDDS prediction columns
nutrient_mapping = {
    'total_calories': 'Energy (kcal)',     # GT column: total_calories -> Pred: Energy (kcal)
    'total_protein': 'Protein (g)',        # GT column: total_protein -> Pred: Protein (g)
    'total_carbs': 'Carbohydrate (g)',     # GT column: total_carbs -> Pred: Carbohydrate (g)
    'total_fat': 'Total Fat (g)',          # GT column: total_fat -> Pred: Total Fat (g)
    'total_mass': 'GPTWeight'              # GT column: total_mass -> Pred: GPTWeight
}

# Calculate MAE for each nutrient
mae_results = {}

for gt_col, pred_col in nutrient_mapping.items():
    # Check if columns exist
    if gt_col not in df.columns:
        print(f"Warning: Ground truth column {gt_col} not found")
        continue
    
    if pred_col not in df.columns:
        print(f"Warning: Prediction column {pred_col} not found")
        continue
    
    # Filter to valid rows (both GT and prediction exist)
    valid_mask = (
        df[gt_col].notna() & 
        df[pred_col].notna()
    )
    
    valid_data = df[valid_mask]
    
    if len(valid_data) == 0:
        print(f"No valid data for {gt_col}")
        continue
    
    # Calculate MAE
    mae = np.abs(valid_data[gt_col] - valid_data[pred_col]).mean()
    
    # Use readable names for display
    display_name = {
        'total_calories': 'Energy (kcal)',
        'total_protein': 'Protein (g)',
        'total_carbs': 'Carbohydrate (g)',
        'total_fat': 'Total Fat (g)',
        'total_mass': 'Food weight (g)'
    }
    
    mae_results[display_name[gt_col]] = {
        'MAE': mae,
        'N': len(valid_data)
    }

# Print formatted table
print("\n" + "="*70)
print("MAE Results for DietAI24 on Nutrition5k Dataset (Per Dish)")
print("="*70)
print(f"{'Target':<25} {'MAE (Per dish)':<20} {'N samples':<15}")
print("-"*70)

for target, results in mae_results.items():
    print(f"{target:<25} {results['MAE']:<20.2f} {results['N']:<15}")

print("="*70)

Total rows in Nutrition5k results: 1000
Columns: ['dish_id', 'total_calories', 'total_mass', 'total_fat', 'total_carbs', 'total_protein', 'ingredients', 'sorted_ingredients', 'GPTFoodDescription', 'GPTFoodCode', 'GPTAmount', 'GPTAmountWeight', 'Energy (kcal)', 'Protein (g)', 'Carbohydrate (g)', 'Total Fat (g)', 'FoodCodeError', 'WeightError', 'IngredientNum', 'GPTWeight']

MAE Results for DietAI24 on Nutrition5k Dataset (Per Dish)
Target                    MAE (Per dish)       N samples      
----------------------------------------------------------------------
Energy (kcal)             186.87               947            
Protein (g)               12.65                947            
Carbohydrate (g)          15.35                947            
Total Fat (g)             11.73                947            
Food weight (g)           121.45               1000           
